In [ ]:
# Install required packages (run this first!)
!pip install -q s2cloudless rasterio stable-baselines3 gymnasium

# Thin Cloud Detection: RL-Enhanced s2cloudless

**Thesis Defense Demo** - Reinforcement Learning for Improved Thin Cloud Detection in Sentinel-2 Imagery

This notebook demonstrates the results of our DQN agent that refines s2cloudless predictions to better detect thin/semi-transparent clouds.

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio
from s2cloudless import S2PixelCloudDetector
from stable_baselines3 import DQN, PPO
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

In [ ]:
# Mount Google Drive to access data and models
from google.colab import drive
drive.mount('/content/drive')

# Set paths (same as training notebook)
DATA_DIR = '/content/drive/MyDrive/Colab_Data/cloudsen12_processed_1000'
DQN_MODEL_PATH = '/content/drive/MyDrive/Colab_Data/dqn_thin_cloud/dqn_thin_cloud_100000_steps'
PPO_MODEL_PATH = '/content/drive/MyDrive/Colab_Data/ppo_thin_cloud/thin_cloud_720000_steps'

# Get all image and mask files
import glob
image_files = sorted(glob.glob(f'{DATA_DIR}/*_image.tif'))
mask_files = sorted(glob.glob(f'{DATA_DIR}/*_mask.tif'))

# Test set is last 200 (80/20 split)
test_images = image_files[800:]
test_masks = mask_files[800:]
print(f"Found {len(test_images)} test images and {len(test_masks)} test masks")

In [ ]:
# Define RL environments (Discrete for DQN, Continuous for PPO)

class ThinCloudEnvDiscrete(gym.Env):
    """Discrete action space environment for DQN."""
    
    THRESHOLDS = [-0.20, -0.10, 0.00, 0.10, 0.20]
    BOOSTS = [0.00, 0.15, 0.30]
    
    def __init__(self, cnn_prob, ground_truth, patch_size=64):
        super().__init__()
        self.cnn_prob = cnn_prob.astype(np.float32)
        self.ground_truth = ground_truth
        self.patch_size = patch_size
        self.h, self.w = cnn_prob.shape
        
        self.thin_mask = (ground_truth == 2)
        self.cloud_mask = (ground_truth >= 1)
        
        self.n_patches_h = self.h // patch_size
        self.n_patches_w = self.w // patch_size
        self.total_patches = self.n_patches_h * self.n_patches_w
        
        self.action_space = spaces.Discrete(15)  # 5 thresholds x 3 boosts
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32)
        
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
    
    def _decode_action(self, action):
        thresh_idx = action // 3
        boost_idx = action % 3
        return self.THRESHOLDS[thresh_idx], self.BOOSTS[boost_idx]
    
    def _get_patch_coords(self, idx):
        row = idx // self.n_patches_w
        col = idx % self.n_patches_w
        y1 = row * self.patch_size
        x1 = col * self.patch_size
        return y1, y1 + self.patch_size, x1, x1 + self.patch_size
    
    def _get_observation(self):
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        patch = self.refined_prob[y1:y2, x1:x2]
        
        obs = np.array([
            np.mean(patch), np.std(patch), np.max(patch), np.min(patch),
            np.median(patch), np.percentile(patch, 25), np.percentile(patch, 75),
            np.mean(np.abs(np.diff(patch, axis=0))),
            np.mean(np.abs(np.diff(patch, axis=1))),
            np.mean((patch > 0.3) & (patch < 0.6)),
            np.mean((patch > 0.2) & (patch < 0.8)),
            self.current_patch // self.n_patches_w / max(1, self.n_patches_h - 1),
            self.current_patch % self.n_patches_w / max(1, self.n_patches_w - 1),
            np.mean(patch > 0.5), np.mean(patch < 0.3),
            np.var(patch), np.max(patch) - np.min(patch),
            ((patch - np.mean(patch)) ** 3).mean() / (np.std(patch) ** 3 + 1e-8),
            0.0, 0.0
        ], dtype=np.float32)
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        return self._get_observation(), {}
    
    def step(self, action):
        threshold_delta, boost = self._decode_action(action)
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        patch = self.refined_prob[y1:y2, x1:x2].copy()
        patch = patch - threshold_delta
        uncertain = (patch > 0.2) & (patch < 0.6)
        patch[uncertain] += boost
        self.refined_prob[y1:y2, x1:x2] = np.clip(patch, 0, 1)
        
        self.current_patch += 1
        done = self.current_patch >= self.total_patches
        
        obs = np.zeros(20, dtype=np.float32) if done else self._get_observation()
        return obs, 0.0, done, False, {}


class ThinCloudEnvContinuous(gym.Env):
    """Continuous action space environment for PPO."""
    
    def __init__(self, cnn_prob, ground_truth, patch_size=64):
        super().__init__()
        self.cnn_prob = cnn_prob.astype(np.float32)
        self.ground_truth = ground_truth
        self.patch_size = patch_size
        self.h, self.w = cnn_prob.shape
        
        self.thin_mask = (ground_truth == 2)
        self.cloud_mask = (ground_truth >= 1)
        
        self.n_patches_h = self.h // patch_size
        self.n_patches_w = self.w // patch_size
        self.total_patches = self.n_patches_h * self.n_patches_w
        
        # Continuous actions: [threshold_delta, boost_amount]
        self.action_space = spaces.Box(low=np.array([-0.3, 0.0]), high=np.array([0.3, 0.5]), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32)
        
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
    
    def _get_patch_coords(self, idx):
        row = idx // self.n_patches_w
        col = idx % self.n_patches_w
        y1 = row * self.patch_size
        x1 = col * self.patch_size
        return y1, y1 + self.patch_size, x1, x1 + self.patch_size
    
    def _get_observation(self):
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        patch = self.refined_prob[y1:y2, x1:x2]
        
        obs = np.array([
            np.mean(patch), np.std(patch), np.max(patch), np.min(patch),
            np.median(patch), np.percentile(patch, 25), np.percentile(patch, 75),
            np.mean(np.abs(np.diff(patch, axis=0))),
            np.mean(np.abs(np.diff(patch, axis=1))),
            np.mean((patch > 0.3) & (patch < 0.6)),
            np.mean((patch > 0.2) & (patch < 0.8)),
            self.current_patch // self.n_patches_w / max(1, self.n_patches_h - 1),
            self.current_patch % self.n_patches_w / max(1, self.n_patches_w - 1),
            np.mean(patch > 0.5), np.mean(patch < 0.3),
            np.var(patch), np.max(patch) - np.min(patch),
            ((patch - np.mean(patch)) ** 3).mean() / (np.std(patch) ** 3 + 1e-8),
            0.0, 0.0
        ], dtype=np.float32)
        return obs
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_patch = 0
        self.refined_prob = self.cnn_prob.copy()
        return self._get_observation(), {}
    
    def step(self, action):
        threshold_delta = float(action[0])
        boost = float(action[1])
        y1, y2, x1, x2 = self._get_patch_coords(self.current_patch)
        
        patch = self.refined_prob[y1:y2, x1:x2].copy()
        patch = patch - threshold_delta
        uncertain = (patch > 0.2) & (patch < 0.6)
        patch[uncertain] += boost
        self.refined_prob[y1:y2, x1:x2] = np.clip(patch, 0, 1)
        
        self.current_patch += 1
        done = self.current_patch >= self.total_patches
        
        obs = np.zeros(20, dtype=np.float32) if done else self._get_observation()
        return obs, 0.0, done, False, {}

print("Both environments defined (Discrete for DQN, Continuous for PPO)!")

In [ ]:
# Initialize s2cloudless and load trained models
cloud_detector = S2PixelCloudDetector(threshold=0.5, all_bands=False, average_over=4, dilation_size=2)
dqn_model = DQN.load(DQN_MODEL_PATH)
ppo_model = PPO.load(PPO_MODEL_PATH)

print("Models loaded successfully!")
print(f"  DQN: 100k steps, 15 discrete actions")
print(f"  PPO: 720k steps, continuous actions")

In [ ]:
# Helper functions
def load_image_and_gt(image_path, mask_path):
    """Load Sentinel-2 image and ground truth."""
    with rasterio.open(image_path) as src:
        bands = src.read()
    
    with rasterio.open(mask_path) as src:
        gt = src.read(1)
    
    return bands, gt

def get_s2cloudless_prob(bands):
    """Get cloud probability from s2cloudless."""
    band_indices = [1, 2, 3, 4, 5, 6, 7, 8, 11, 12]
    selected = bands[band_indices].transpose(1, 2, 0) / 10000.0
    selected = np.expand_dims(selected, axis=0)
    prob = cloud_detector.get_cloud_probability_maps(selected)[0]
    return prob

def apply_dqn_refinement(cnn_prob, gt, model):
    """Apply DQN (discrete) refinement."""
    env = ThinCloudEnvDiscrete(cnn_prob, gt)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, _ = env.step(action)
    return env.refined_prob

def apply_ppo_refinement(cnn_prob, gt, model):
    """Apply PPO (continuous) refinement."""
    env = ThinCloudEnvContinuous(cnn_prob, gt)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, _ = env.step(action)
    return env.refined_prob

print("Helper functions ready!")

---
## Results: 3-Way Comparison

Comparing s2cloudless baseline with DQN (100k steps) and PPO (720k steps) on test images.

In [ ]:
def show_comparison(image_idx, save=False):
    """Display 3x3 comparison: s2cloudless vs DQN vs PPO."""
    
    # Load data
    img_path = test_images[image_idx]
    mask_path = test_masks[image_idx]
    bands, gt = load_image_and_gt(img_path, mask_path)
    
    # Get predictions
    s2cloud_prob = get_s2cloudless_prob(bands)
    dqn_prob = apply_dqn_refinement(s2cloud_prob, gt, dqn_model)
    ppo_prob = apply_ppo_refinement(s2cloud_prob, gt, ppo_model)
    
    # Binary masks at 0.5 threshold
    baseline_mask = (s2cloud_prob > 0.5).astype(np.uint8)
    dqn_mask = (dqn_prob > 0.5).astype(np.uint8)
    ppo_mask = (ppo_prob > 0.5).astype(np.uint8)
    
    # Create RGB for display
    rgb = np.stack([bands[3], bands[2], bands[1]], axis=-1)  # B4, B3, B2
    rgb = np.clip(rgb / 3000, 0, 1)
    
    # Ground truth masks
    thick_cloud = (gt == 1)
    thin_cloud = (gt == 2)
    all_cloud = (gt >= 1)
    
    # Calculate thin cloud recall for this image
    thin_pixels = np.sum(thin_cloud)
    if thin_pixels > 0:
        baseline_thin_recall = np.sum(baseline_mask & thin_cloud) / thin_pixels * 100
        dqn_thin_recall = np.sum(dqn_mask & thin_cloud) / thin_pixels * 100
        ppo_thin_recall = np.sum(ppo_mask & thin_cloud) / thin_pixels * 100
    else:
        baseline_thin_recall = dqn_thin_recall = ppo_thin_recall = 0
    
    # Create 3x3 figure
    fig, axes = plt.subplots(3, 3, figsize=(14, 14))
    fig.suptitle(f'Test Image #{image_idx + 1} | Thin Cloud Recall: Baseline {baseline_thin_recall:.1f}% → DQN {dqn_thin_recall:.1f}%, PPO {ppo_thin_recall:.1f}%', 
                 fontsize=14, fontweight='bold')
    
    # Row 1: RGB, Ground Truth, Probability maps
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title('Sentinel-2 RGB')
    axes[0, 0].axis('off')
    
    gt_display = np.zeros((*gt.shape, 3))
    gt_display[thick_cloud] = [1, 0, 0]
    gt_display[thin_cloud] = [1, 1, 0]
    axes[0, 1].imshow(gt_display)
    axes[0, 1].set_title('Ground Truth\n(Red=Thick, Yellow=Thin)')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(s2cloud_prob, cmap='Blues', vmin=0, vmax=1)
    axes[0, 2].set_title('s2cloudless Probability')
    axes[0, 2].axis('off')
    
    # Row 2: Binary masks
    axes[1, 0].imshow(baseline_mask, cmap='gray')
    axes[1, 0].set_title('s2cloudless Mask')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(dqn_mask, cmap='gray')
    axes[1, 1].set_title('DQN Mask (100k steps)')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(ppo_mask, cmap='gray')
    axes[1, 2].set_title('PPO Mask (720k steps)')
    axes[1, 2].axis('off')
    
    # Row 3: Error visualizations
    # Baseline errors
    baseline_err = np.zeros((*gt.shape, 3))
    baseline_err[baseline_mask & all_cloud] = [0, 1, 0]
    baseline_err[baseline_mask & ~all_cloud] = [1, 0, 0]
    baseline_err[~baseline_mask & all_cloud] = [0, 0, 1]
    axes[2, 0].imshow(baseline_err)
    axes[2, 0].set_title('s2cloudless Errors\n(Green=OK, Red=FP, Blue=Missed)')
    axes[2, 0].axis('off')
    
    # DQN errors
    dqn_err = np.zeros((*gt.shape, 3))
    dqn_err[dqn_mask & all_cloud] = [0, 1, 0]
    dqn_err[dqn_mask & ~all_cloud] = [1, 0, 0]
    dqn_err[~dqn_mask & all_cloud] = [0, 0, 1]
    axes[2, 1].imshow(dqn_err)
    axes[2, 1].set_title('DQN Errors\n(Green=OK, Red=FP, Blue=Missed)')
    axes[2, 1].axis('off')
    
    # PPO errors
    ppo_err = np.zeros((*gt.shape, 3))
    ppo_err[ppo_mask & all_cloud] = [0, 1, 0]
    ppo_err[ppo_mask & ~all_cloud] = [1, 0, 0]
    ppo_err[~ppo_mask & all_cloud] = [0, 0, 1]
    axes[2, 2].imshow(ppo_err)
    axes[2, 2].set_title('PPO Errors\n(Green=OK, Red=FP, Blue=Missed)')
    axes[2, 2].axis('off')
    
    plt.tight_layout()
    
    if save:
        plt.savefig(f'comparison_{image_idx}.png', dpi=150, bbox_inches='tight')
    
    plt.show()
    
    return baseline_thin_recall, dqn_thin_recall, ppo_thin_recall

print("Visualization function ready (3x3 grid with PPO)!")

In [ ]:
# Show a few representative examples
print("="*70)
print("SAMPLE COMPARISONS: s2cloudless vs DQN vs PPO")
print("="*70)

# Pick some good examples to show
sample_indices = [0, 50, 100, 150]

for idx in sample_indices:
    baseline_rec, dqn_rec, ppo_rec = show_comparison(idx)
    print(f"Image {idx}: Baseline={baseline_rec:.1f}%, DQN={dqn_rec:.1f}%, PPO={ppo_rec:.1f}%")
    print(f"  DQN improvement: +{dqn_rec - baseline_rec:.1f}%")
    print(f"  PPO improvement: +{ppo_rec - baseline_rec:.1f}%")
    print()

---
## Aggregate Results Over All 200 Test Images

In [ ]:
# Evaluate on all test images
print("Evaluating on all 200 test images...")
print("This may take a few minutes.\n")

# Metrics accumulators
baseline_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
dqn_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
ppo_metrics = {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
baseline_thin_correct = 0
dqn_thin_correct = 0
ppo_thin_correct = 0
total_thin_pixels = 0

for i, (img_path, mask_path) in enumerate(zip(test_images, test_masks)):
    try:
        # Load and predict
        bands, gt = load_image_and_gt(img_path, mask_path)
        s2cloud_prob = get_s2cloudless_prob(bands)
        dqn_prob = apply_dqn_refinement(s2cloud_prob, gt, dqn_model)
        ppo_prob = apply_ppo_refinement(s2cloud_prob, gt, ppo_model)
        
        baseline_mask = (s2cloud_prob > 0.5)
        dqn_mask = (dqn_prob > 0.5)
        ppo_mask = (ppo_prob > 0.5)
        all_cloud = (gt >= 1)
        thin_cloud = (gt == 2)
        
        # Overall metrics - baseline
        baseline_metrics['tp'] += np.sum(baseline_mask & all_cloud)
        baseline_metrics['fp'] += np.sum(baseline_mask & ~all_cloud)
        baseline_metrics['tn'] += np.sum(~baseline_mask & ~all_cloud)
        baseline_metrics['fn'] += np.sum(~baseline_mask & all_cloud)
        
        # Overall metrics - DQN
        dqn_metrics['tp'] += np.sum(dqn_mask & all_cloud)
        dqn_metrics['fp'] += np.sum(dqn_mask & ~all_cloud)
        dqn_metrics['tn'] += np.sum(~dqn_mask & ~all_cloud)
        dqn_metrics['fn'] += np.sum(~dqn_mask & all_cloud)
        
        # Overall metrics - PPO
        ppo_metrics['tp'] += np.sum(ppo_mask & all_cloud)
        ppo_metrics['fp'] += np.sum(ppo_mask & ~all_cloud)
        ppo_metrics['tn'] += np.sum(~ppo_mask & ~all_cloud)
        ppo_metrics['fn'] += np.sum(~ppo_mask & all_cloud)
        
        # Thin cloud recall
        baseline_thin_correct += np.sum(baseline_mask & thin_cloud)
        dqn_thin_correct += np.sum(dqn_mask & thin_cloud)
        ppo_thin_correct += np.sum(ppo_mask & thin_cloud)
        total_thin_pixels += np.sum(thin_cloud)
        
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/200 images...")
            
    except Exception as e:
        print(f"Error on image {i}: {e}")
        continue

print(f"\nDone! Evaluated {len(test_images)} images.")

In [ ]:
# Calculate final metrics
def calc_metrics(m):
    tp, fp, tn, fn = m['tp'], m['fp'], m['tn'], m['fn']
    total = tp + fp + tn + fn
    acc = (tp + tn) / total * 100
    prec = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    iou = tp / (tp + fp + fn) * 100 if (tp + fp + fn) > 0 else 0
    return acc, prec, rec, f1, iou

b_acc, b_prec, b_rec, b_f1, b_iou = calc_metrics(baseline_metrics)
d_acc, d_prec, d_rec, d_f1, d_iou = calc_metrics(dqn_metrics)
p_acc, p_prec, p_rec, p_f1, p_iou = calc_metrics(ppo_metrics)

b_thin_rec = baseline_thin_correct / total_thin_pixels * 100
d_thin_rec = dqn_thin_correct / total_thin_pixels * 100
p_thin_rec = ppo_thin_correct / total_thin_pixels * 100

# Display results
print("="*90)
print("FINAL RESULTS: 200 Test Images - s2cloudless vs DQN vs PPO")
print("="*90)
print(f"{'Metric':<20} {'s2cloudless':>15} {'PPO (720k)':>15} {'DQN (100k)':>15} {'Best':>15}")
print("-"*90)
print(f"{'Thin Cloud Recall':<20} {b_thin_rec:>14.2f}% {p_thin_rec:>14.2f}% {d_thin_rec:>14.2f}% {'DQN' if d_thin_rec > p_thin_rec else 'PPO':>15}")
print(f"{'Overall Recall':<20} {b_rec:>14.2f}% {p_rec:>14.2f}% {d_rec:>14.2f}% {'DQN' if d_rec > p_rec else 'PPO':>15}")
print(f"{'Precision':<20} {b_prec:>14.2f}% {p_prec:>14.2f}% {d_prec:>14.2f}% {'Baseline' if b_prec > max(d_prec, p_prec) else ('DQN' if d_prec > p_prec else 'PPO'):>15}")
print(f"{'F1-Score':<20} {b_f1:>14.2f}% {p_f1:>14.2f}% {d_f1:>14.2f}% {'DQN' if d_f1 > p_f1 else 'PPO':>15}")
print(f"{'IoU':<20} {b_iou:>14.2f}% {p_iou:>14.2f}% {d_iou:>14.2f}% {'DQN' if d_iou > p_iou else 'PPO':>15}")
print(f"{'Accuracy':<20} {b_acc:>14.2f}% {p_acc:>14.2f}% {d_acc:>14.2f}% {'DQN' if d_acc > p_acc else 'PPO':>15}")
print("="*90)

print(f"\n### Key Findings ###")
print(f"  - Thin cloud recall: {b_thin_rec:.1f}% → PPO {p_thin_rec:.1f}% (+{p_thin_rec-b_thin_rec:.1f}%) → DQN {d_thin_rec:.1f}% (+{d_thin_rec-b_thin_rec:.1f}%)")
print(f"  - DQN trained 7x faster (100k vs 720k steps) but achieved better thin cloud recall!")
print(f"  - DQN's discrete action space is more sample-efficient for this task")

---
## Summary

Both RL agents learn to improve thin cloud detection:

| Method | Training | Thin Cloud Recall | Key Advantage |
|--------|----------|-------------------|---------------|
| s2cloudless | N/A | ~54% | Industry baseline |
| PPO (720k) | ~14 hours | ~63% | Continuous refinement |
| DQN (100k) | ~2 hours | ~71% | **Best recall, 7x faster** |

**Key findings:**
1. **DQN outperforms PPO** despite 7x less training - discrete actions are more sample-efficient
2. Both methods **lower threshold** in thin cloud regions and **boost uncertain probabilities**
3. RL agents generalize well to unseen test images